In [1]:
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).
Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


# Viewers who view multiple articles 

In [2]:
import pandas as pd
import numpy as np
from random import randint, choice, seed

seed(42)
np.random.seed(42)

n_rows = 100

viewer_ids = np.random.randint(1, 21, n_rows)
article_ids = np.random.randint(1, 31, n_rows)
author_ids = np.random.randint(1, 11, n_rows)

dates = pd.date_range("2019-07-20", "2019-08-10")
view_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

views = pd.DataFrame({
    "article_id": article_ids,
    "author_id": author_ids,
    "viewer_id": viewer_ids,
    "view_date": view_dates
})

# Inject guaranteed qualifying cases
extra = pd.DataFrame([
    [101, 1, 5, "2019-08-01"],
    [102, 2, 5, "2019-08-01"],
    [201, 3, 6, "2019-08-02"],
    [202, 4, 6, "2019-08-02"],
])

extra.columns = views.columns

# Add duplicate rows
duplicates = pd.DataFrame([
    [101, 1, 5, "2019-08-01"],
    [202, 4, 6, "2019-08-02"],
])

duplicates.columns = views.columns

views = pd.concat([views, extra, duplicates], ignore_index=True)

print(views.head())
print("\nShape:", views.shape)

   article_id  author_id  viewer_id   view_date
0           3          4          7  2019-07-23
1          12          7         20  2019-07-20
2           8          2         15  2019-08-02
3          22          3         11  2019-08-09
4          27          1          8  2019-08-04

Shape: (106, 4)


In [4]:
sdf = spark.createDataFrame(views)

In [5]:
display(sdf)

,article_id,author_id,viewer_id,view_date
0,3,4,7,2019-07-23
1,12,7,20,2019-07-20
2,8,2,15,2019-08-02
3,22,3,11,2019-08-09
4,27,1,8,2019-08-04


In [6]:
sdf.createOrReplaceTempView("views")

In [29]:
display(spark.sql("""SELECT DISTINCT viewer_id
FROM views
GROUP BY viewer_id, view_date
HAVING COUNT(DISTINCT article_id) > 1
ORDER BY viewer_id"""));

,viewer_id
0,2
1,4
2,5
3,6
4,7


In [7]:
spark.sql("select * from views")

DataFrame[article_id: bigint, author_id: bigint, viewer_id: bigint, view_date: string]

In [43]:
from pyspark.sql.functions import col, countDistinct

display(
    sdf.groupBy("viewer_id", "view_date")
       .agg(countDistinct("article_id").alias("article_id_dist"))
       .filter(col("article_id_dist") > 1)
       .select("viewer_id")
       .distinct()
       .orderBy("viewer_id")
)

,viewer_id
0,2
1,4
2,5
3,6
4,7


In [44]:
display(
    sdf.groupBy("viewer_id", "view_date")
       .agg(countDistinct("article_id").alias("article_id_dist")))

,viewer_id,view_date,article_id_dist
0,8,2019-07-22,1
1,14,2019-07-31,1
2,16,2019-07-30,1
3,1,2019-08-08,1
4,8,2019-08-04,1


# pattern behind this problem

In [ ]:
df.groupBy(...)
  .agg(...)
  .filter(...)
  .select(...)
  .distinct()
  .orderBy(...)\

# Another example on same pattern

In [51]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 100

customer_ids = np.random.randint(100, 121, n_rows)
product_ids = np.random.randint(1, 31, n_rows)
order_ids = np.arange(1, n_rows + 1)

dates = pd.date_range("2024-01-01", "2024-01-10")
order_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

orders = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": customer_ids,
    "product_id": product_ids,
    "order_date": order_dates
})

# Inject qualifying customers
extra = pd.DataFrame([
    [1001, 101, 101, "2024-01-05"],
    [1002, 101, 102, "2024-01-05"],
    [1003, 105, 201, "2024-01-08"],
    [1004, 105, 202, "2024-01-08"],
], columns=orders.columns)

# Add duplicate orders
duplicates = pd.DataFrame([
    [1005, 101, 101, "2024-01-05"],
    [1006, 105, 202, "2024-01-08"],
], columns=orders.columns)

orders = pd.concat([orders, extra, duplicates], ignore_index=True)

orders.head()



AttributeError: 'SparkSession' object has no attribute 'Dataframe'

## Find all customers who ordered more than one distinct product on the same day.

In [ ]:
sdf = spark.createDataFrame(orders)

In [54]:
sdf.createOrReplaceTempView("orders")

In [58]:
display(spark.sql("""SELECT Distinct(customer_id) from orders group by customer_id,order_date having count(distinct(product_id))>1 """))

,customer_id
0,110
1,119
2,107
3,103
4,114


In [55]:
display(sdf)

,order_id,customer_id,product_id,order_date
0,1,106,28,2024-01-10
1,2,119,7,2024-01-08
2,3,114,9,2024-01-06
3,4,110,8,2024-01-08
4,5,107,12,2024-01-09


In [64]:
from pyspark.sql.functions import col, countDistinct

display(
    sdf.groupBy("customer_id", "order_date")
       .agg(countDistinct("product_id").alias("product_id_dist"))
       .filter(col("product_id_dist") > 1)
       .select("customer_id")
       .distinct()
       .orderBy("customer_id")
)

,customer_id
0,101
1,103
2,104
3,105
4,106


# Pattern: Aggregate → Rank → Bucket → Label

## When to recognize this pattern

These problems usually contain phrases like:

* Top N%
* Bottom N%
* Quartile
* Percentile
* NTILE
* Spending Tier
* Customer Segment
* Bronze / Silver / Gold / Platinum
* Divide into equal groups
* Highest spenders
* Lowest performers

---

# Generic Workflow

```
Raw Data
    ↓
Aggregate
    ↓
Sort
    ↓
Assign Bucket (NTILE / qcut)
    ↓
Map Bucket → Category
    ↓
Return Required Columns
```

---

# Step 1: Aggregate

Determine what needs to be calculated first.

Examples:

* Total spending per customer
* Total sales per salesperson
* Total revenue per store
* Total orders per user

SQL

```sql
GROUP BY entity
SUM(...)
```

PySpark

```python
.groupBy(entity)
.agg(sum(...))
```

Pandas

```python
.groupby(entity)
.agg(...)
```

---

# Step 2: Sort

Identify whether ranking is:

* Highest first (DESC)
* Lowest first (ASC)

Example

```
ORDER BY total_spending DESC
```

---

# Step 3: Divide into Buckets

Common methods

### SQL

```
NTILE(4)
```

### PySpark

```
ntile(4).over(window)
```

### Pandas

```
pd.qcut(...)
```

---

# Step 4: Map Bucket to Labels

Example

| Bucket | Tier     |
| ------ | -------- |
| 1      | Platinum |
| 2      | Gold     |
| 3      | Silver   |
| 4      | Bronze   |

---

# Step 5: Return Output

Return only the required columns.

Example

```
customer_id
total_spending
tier
```

Sort as requested.

---

# SQL Pattern

```sql
WITH totals AS (
    SELECT
        entity,
        SUM(metric) AS total_metric
    FROM table
    GROUP BY entity
),
ranked AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY total_metric DESC) AS bucket
    FROM totals
)
SELECT ...
FROM ranked;
```

---

# PySpark Pattern

```python
aggregate
    ↓
Window.orderBy(...)
    ↓
ntile(...)
    ↓
withColumn(...)
    ↓
select(...)
```

---

# Pandas Pattern

```python
groupby()
    ↓
sum()
    ↓
sort_values()
    ↓
qcut()
    ↓
map()
    ↓
return
```

---

# Mental Checklist

Before writing code, answer these questions:

1. What is the entity?
2. What metric should be aggregated?
3. Should it be SUM, AVG, COUNT, etc.?
4. Should ranking be ASC or DESC?
5. How many buckets are required?
6. Which function creates the buckets? (NTILE / qcut)
7. How are buckets mapped to labels?
8. Which columns should be returned?
9. Is final sorting required?

---

# Common Interview Problems

* Customer Spending Tiers
* Employee Salary Bands
* Product Revenue Quartiles
* Store Performance Levels
* Customer Loyalty Segments
* Salesperson Rankings
* Department Performance Categories
* Student Score Percentiles
* Vendor Rating Tiers
* Branch Profit Categories

---

# Recognition Rule

Whenever a problem says:

> Aggregate a metric → Rank the results → Divide into equal groups → Assign labels

it is almost always an **Aggregate → Rank → Bucket → Label** problem using:

* SQL: `SUM()` + `NTILE()`
* PySpark: `Window` + `ntile()`
* Pandas: `groupby()` + `qcut()`


In [65]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 200
n_customers = 40

purchase_ids = np.arange(1, n_rows + 1)
customer_ids = np.random.randint(1, n_customers + 1, n_rows)
amounts = np.round(np.random.uniform(20, 2500, n_rows), 2)

dates = pd.date_range("2023-01-01", "2023-06-30")
purchase_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

purchases = pd.DataFrame({
    "purchase_id": purchase_ids,
    "customer_id": customer_ids,
    "amount": amounts,
    "purchase_date": purchase_dates
})

# Inject customers with clearly different spending totals
extra = pd.DataFrame({
    "purchase_id": np.arange(n_rows + 1, n_rows + 17),
    "customer_id": [
        101,101,
        102,102,
        103,103,
        104,104,
        105,105,
        106,106,
        107,107,
        108,108
    ],
    "amount": [
        6000,5500,   # 101
        5000,4500,   # 102
        4000,3500,   # 103
        3000,2800,   # 104
        2000,1800,   # 105
        1200,1000,   # 106
        700,600,     # 107
        150,100      # 108
    ],
    "purchase_date": [
        "2023-05-01","2023-06-01",
        "2023-05-02","2023-06-02",
        "2023-05-03","2023-06-03",
        "2023-05-04","2023-06-04",
        "2023-05-05","2023-06-05",
        "2023-05-06","2023-06-06",
        "2023-05-07","2023-06-07",
        "2023-05-08","2023-06-08"
    ]
})

purchases = pd.concat([purchases, extra], ignore_index=True)

print(purchases.head())
print("\nShape:", purchases.shape)

   purchase_id  customer_id   amount purchase_date
0            1           39   815.53    2023-06-10
1            2           29   482.57    2023-06-17
2            3           15   121.12    2023-03-15
3            4            8  1485.41    2023-02-12
4            5           21  1700.36    2023-02-13

Shape: (216, 4)


In [67]:
purchases = spark.createDataFrame(purchases)

In [68]:
purchases.createOrReplaceTempView("purchases")

In [77]:
display(spark.sql("""
WITH totals AS (
    SELECT
        customer_id,
        SUM(amount) AS total_spending
    FROM purchases
    GROUP BY customer_id
),
ranked AS (
    SELECT
        customer_id,
        total_spending,
        NTILE(4) OVER (ORDER BY total_spending DESC) AS bucket
    FROM totals
)
SELECT
    customer_id,
    total_spending,
    CASE bucket
        WHEN 1 THEN 'Platinum'
        WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver'
        WHEN 4 THEN 'Bronze'
    END AS tier
FROM ranked
ORDER BY total_spending DESC;
"""))

,customer_id,total_spending,tier
0,24,13429.87,Platinum
1,33,12005.71,Platinum
2,28,11957.36,Platinum
3,101,11500.00,Platinum
4,37,10727.12,Platinum


# Another Problem on this pattern

In [78]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 250
n_salespersons = 40

sale_ids = np.arange(1, n_rows + 1)
salesperson_ids = np.random.randint(1, n_salespersons + 1, n_rows)
product_ids = np.random.randint(100, 151, n_rows)
quantities = np.random.randint(1, 10, n_rows)
unit_prices = np.round(np.random.uniform(100, 5000, n_rows), 2)

dates = pd.date_range("2024-01-01", "2024-03-31")
sale_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

sales = pd.DataFrame({
    "sale_id": sale_ids,
    "salesperson_id": salesperson_ids,
    "product_id": product_ids,
    "quantity": quantities,
    "unit_price": unit_prices,
    "sale_date": sale_dates
})

# Inject high/low performers
extra = pd.DataFrame({
    "sale_id": np.arange(n_rows + 1, n_rows + 17),
    "salesperson_id": [
        101,101,
        102,102,
        103,103,
        104,104,
        105,105,
        106,106,
        107,107,
        108,108
    ],
    "product_id": [
        201,202,
        203,204,
        205,206,
        207,208,
        209,210,
        211,212,
        213,214,
        215,216
    ],
    "quantity": [
        20,18,
        18,17,
        15,15,
        12,11,
        8,8,
        6,5,
        4,3,
        1,1
    ],
    "unit_price": [
        6000,5500,
        5000,4800,
        4000,3900,
        3000,2800,
        2200,2100,
        1500,1400,
        900,850,
        200,150
    ],
    "sale_date": [
        "2024-03-01","2024-03-02",
        "2024-03-03","2024-03-04",
        "2024-03-05","2024-03-06",
        "2024-03-07","2024-03-08",
        "2024-03-09","2024-03-10",
        "2024-03-11","2024-03-12",
        "2024-03-13","2024-03-14",
        "2024-03-15","2024-03-16"
    ]
})

sales = pd.concat([sales, extra], ignore_index=True)

print(sales.head())
print("\nShape:", sales.shape)

   sale_id  salesperson_id  product_id  quantity  unit_price   sale_date
0        1              39         134         2     3518.57  2024-03-25
1        2              29         147         3      982.33  2024-02-18
2        3              15         124         2     3512.86  2024-03-11
3        4               8         134         3     2117.14  2024-03-21
4        5              21         124         7     4384.16  2024-03-24

Shape: (266, 6)


In [80]:
sales = spark.createDataFrame(sales)

In [81]:
display(sales)

,sale_id,salesperson_id,product_id,quantity,unit_price,sale_date
0,1,39,134,2,3518.57,2024-03-25
1,2,29,147,3,982.33,2024-02-18
2,3,15,124,2,3512.86,2024-03-11
3,4,8,134,3,2117.14,2024-03-21
4,5,21,124,7,4384.16,2024-03-24


In [82]:
sales.createOrReplaceTempView("sales")

In [85]:
display(spark.sql("""
with total as (
    SELECT salesperson_id,
            sum(unit_price * quantity)  as total_revenue  
            from sales 
    group by  
    salesperson_id 
), 
ranked as (
    SELECT salesperson_id,total_revenue, 
    Ntile(4) over(order by total_revenue DESC) as bucket 
    from total 
)

SELECT salesperson_id, 
       total_revenue,
        CASE bucket
        WHEN 1 THEN 'Elite'
        WHEN 2 THEN 'High'
        WHEN 3 THEN 'Medium'
        WHEN 4 THEN 'Low'
    END AS performance_level
    from ranked Order by total_revenue DESC
"""))

,salesperson_id,total_revenue,performance_level
0,101,219000.00,Elite
1,24,184286.01,Elite
2,102,171600.00,Elite
3,3,147321.45,Elite
4,37,136271.06,Elite


# another problem on same pattern

In [86]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 300
n_stores = 50

transaction_ids = np.arange(1, n_rows + 1)
store_ids = np.random.randint(1, n_stores + 1, n_rows)
quantities = np.random.randint(1, 20, n_rows)
unit_prices = np.round(np.random.uniform(50, 2000, n_rows), 2)

dates = pd.date_range("2024-01-01", "2024-06-30")
transaction_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

transactions = pd.DataFrame({
    "transaction_id": transaction_ids,
    "store_id": store_ids,
    "quantity": quantities,
    "unit_price": unit_prices,
    "transaction_date": transaction_dates
})

# Inject stores with clearly different revenues
extra = pd.DataFrame({
    "transaction_id": np.arange(n_rows + 1, n_rows + 17),
    "store_id": [
        101,101,
        102,102,
        103,103,
        104,104,
        105,105,
        106,106,
        107,107,
        108,108
    ],
    "quantity": [
        25,20,
        22,20,
        18,18,
        15,14,
        10,10,
        8,7,
        5,4,
        2,1
    ],
    "unit_price": [
        9000,8500,
        7500,7200,
        6000,5800,
        4500,4200,
        3000,2800,
        1800,1700,
        900,850,
        200,150
    ],
    "transaction_date": [
        "2024-06-01","2024-06-02",
        "2024-06-03","2024-06-04",
        "2024-06-05","2024-06-06",
        "2024-06-07","2024-06-08",
        "2024-06-09","2024-06-10",
        "2024-06-11","2024-06-12",
        "2024-06-13","2024-06-14",
        "2024-06-15","2024-06-16"
    ]
})

transactions = pd.concat([transactions, extra], ignore_index=True)

print(transactions.head())
print("\nShape:", transactions.shape)

   transaction_id  store_id  quantity  unit_price transaction_date
0               1        39         4     1787.73       2024-01-15
1               2        29         6     1079.02       2024-06-05
2               3        15         8     1986.28       2024-05-15
3               4        43         3      193.90       2024-01-05
4               5         8        16     1130.02       2024-06-05

Shape: (316, 5)


In [87]:
df = spark.createDataFrame(transactions)

In [90]:
df.createOrReplaceTempView("transactions")

In [93]:
display(spark.sql(
"""
with total as 
(  
    SELECT 
        store_id,
            sum(unit_price * quantity) as total_revenue 
        from transactions 
    GROUP BY store_id      
),
ranked as (
        SELECT
            store_id, 
                total_revenue,
            NTILE(5) OVER(ORDER BY total_revenue DESC) as bucket
        from total
)

SELECT 
    store_id,
        total_revenue, 
            CASE bucket 
                WHEN 1 THEN "Diamond"
                WHEN 2 THEN "Platinum"
                WHEN 3 THEN "Gold"
                WHEN 4 THEN "SILVER"
                WHEN 5 THEN "BRONZE"
                    End as revenue_band
        from ranked
    ORDER BY total_revenue DESC
                


"""))

,store_id,total_revenue,revenue_band
0,101,395000.00,Diamond
1,102,309000.00,Diamond
2,103,212400.00,Diamond
3,49,157847.12,Diamond
4,104,126300.00,Diamond


# Pattern: Aggregate → Window Function → Compare 

# Problem - Active Users MOM

In [94]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 500
n_users = 120

user_ids = np.random.randint(1, n_users + 1, n_rows)

dates = pd.date_range("2023-01-01", "2023-06-30")
login_dates = np.random.choice(
    dates.strftime("%Y-%m-%d"),
    n_rows
)

logins = pd.DataFrame({
    "user_id": user_ids,
    "login_date": login_dates
})

# Inject users to create predictable month-over-month activity
extra = pd.DataFrame({
    "user_id": [
        # January
        201,202,203,204,205,
        # February
        201,202,203,204,205,206,207,
        # March
        201,202,203,208,
        # April
        201,202,203,204,205,206,207,208,209,
        # May
        201,202,203,
        # June
        201,202,203,204,205,206
    ],
    "login_date": [
        "2023-01-05","2023-01-10","2023-01-15","2023-01-20","2023-01-25",
        "2023-02-03","2023-02-06","2023-02-12","2023-02-18","2023-02-22","2023-02-25","2023-02-27",
        "2023-03-04","2023-03-10","2023-03-18","2023-03-24",
        "2023-04-02","2023-04-05","2023-04-09","2023-04-12","2023-04-15","2023-04-18","2023-04-21","2023-04-24","2023-04-28",
        "2023-05-03","2023-05-11","2023-05-19",
        "2023-06-02","2023-06-07","2023-06-12","2023-06-18","2023-06-23","2023-06-28"
    ]
})

# Add duplicate logins (same user, same day)
duplicates = pd.DataFrame({
    "user_id": [201, 201, 206, 208],
    "login_date": [
        "2023-02-03",
        "2023-02-03",
        "2023-04-18",
        "2023-04-24"
    ]
})

logins = pd.concat([logins, extra, duplicates], ignore_index=True)

print(logins.head())
print("\nShape:", logins.shape)

   user_id  login_date
0      103  2023-01-13
1       52  2023-03-01
2       93  2023-05-15
3       15  2023-02-26
4      107  2023-02-05

Shape: (538, 2)


In [96]:
df = spark.createDataFrame(logins)

In [97]:
df.createOrReplaceTempView("logins")

In [100]:
display(spark.sql("""
WITH monthly_users AS (
    SELECT
        date_format(login_date, 'yyyy-MM') AS month,
        COUNT(DISTINCT user_id) AS active_users
    FROM logins
    GROUP BY date_format(login_date, 'yyyy-MM')
),
monthly_change AS (
    SELECT
        month,
        active_users,
        LAG(active_users) OVER (ORDER BY month) AS previous_month_users
    FROM monthly_users
)
SELECT
    month,
    active_users,
    ROUND(
        ((active_users - previous_month_users) / previous_month_users) * 100,
        1
    ) AS mom_change_pct
FROM monthly_change
ORDER BY month;
"""))

,month,active_users,mom_change_pct
0,2023-01,64,NaN
1,2023-02,62,-3.1
2,2023-03,64,3.2
3,2023-04,72,12.5
4,2023-05,72,0.0


In [103]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    date_format,
    lag,
    round
)
from pyspark.sql.window import Window

window = Window.orderBy("month")

display(
    df
        .withColumn("month", date_format("login_date", "yyyy-MM"))
        .groupBy("month")
        .agg(countDistinct("user_id").alias("active_users"))
        .withColumn(
            "previous_month_users",
            lag("active_users").over(window)
        )
        .withColumn(
            "mom_change_pct",
            round(
                (
                    (col("active_users") - col("previous_month_users"))
                    / col("previous_month_users")
                ) * 100,
                1
            )
        )
        .select("month", "active_users", "mom_change_pct")
        .orderBy("month")
)

,month,active_users,mom_change_pct
0,2023-01,64,NaN
1,2023-02,62,-3.1
2,2023-03,64,3.2
3,2023-04,72,12.5
4,2023-05,72,0.0


# product sales in First Year

In [104]:
import pandas as pd
import numpy as np

np.random.seed(42)

# ----------------------------
# Product Table
# ----------------------------
n_products = 30

product = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "product_name": [f"Product_{i}" for i in range(1, n_products + 1)]
})

# ----------------------------
# Sales Table
# ----------------------------
n_sales = 200

sales = pd.DataFrame({
    "sale_id": np.arange(1, n_sales + 1),
    "product_id": np.random.randint(1, n_products + 1, n_sales),
    "year": np.random.choice([2019, 2020, 2021, 2022, 2023], n_sales),
    "quantity": np.random.randint(1, 500, n_sales),
    "price": np.round(np.random.uniform(5, 500, n_sales), 2)
})

# ----------------------------------------
# Inject products with multiple first-year sales
# ----------------------------------------
extra_sales = pd.DataFrame({
    "sale_id": np.arange(n_sales + 1, n_sales + 13),
    "product_id": [
        101,101,101,
        102,102,
        103,103,
        104,
        105,105,
        106,106
    ],
    "year": [
        2020,2020,2021,
        2019,2020,
        2021,2022,
        2023,
        2020,2020,
        2019,2021
    ],
    "quantity": [
        120,80,200,
        50,75,
        40,90,
        150,
        100,60,
        30,55
    ],
    "price": [
        19.99,18.49,17.99,
        49.99,45.99,
        99.99,89.99,
        149.99,
        29.99,31.99,
        9.99,8.99
    ]
})

extra_products = pd.DataFrame({
    "product_id": [101,102,103,104,105,106],
    "product_name": [
        "Laptop",
        "Keyboard",
        "Monitor",
        "Printer",
        "Mouse",
        "Speaker"
    ]
})

product = pd.concat([product, extra_products], ignore_index=True)
sales = pd.concat([sales, extra_sales], ignore_index=True)

print(product.head())
print(product.shape)

print()

print(sales.head())
print(sales.shape)

   product_id product_name
0           1    Product_1
1           2    Product_2
2           3    Product_3
3           4    Product_4
4           5    Product_5
(36, 2)

   sale_id  product_id  year  quantity   price
0        1           7  2023       247  311.04
1        2          20  2021        26  171.59
2        3          29  2023       355  329.58
3        4          15  2022       306  195.77
4        5          11  2023       409  342.40
(212, 5)


In [108]:
sales_df = spark.createDataFrame(sales)

In [109]:
display(sales_df)

,sale_id,product_id,year,quantity,price
0,1,7,2023,247,311.04
1,2,20,2021,26,171.59
2,3,29,2023,355,329.58
3,4,15,2022,306,195.77
4,5,11,2023,409,342.40


In [110]:
sales_df.createOrReplaceTempView("sales")

In [112]:
spark.sql("SELECT * FROM sales").show()

+-------+----------+----+--------+------+
|sale_id|product_id|year|quantity| price|
+-------+----------+----+--------+------+
|      1|         7|2023|     247|311.04|
|      2|        20|2021|      26|171.59|
|      3|        29|2023|     355|329.58|
|      4|        15|2022|     306|195.77|
|      5|        11|2023|     409| 342.4|
|      6|         8|2021|     408|173.61|
|      7|        29|2021|      13|134.04|
|      8|        21|2022|     316|250.54|
|      9|         7|2020|     391|347.98|
|     10|        26|2020|     313|177.43|
|     11|        19|2023|      36|468.64|
|     12|        23|2019|     173|  24.4|
|     13|        11|2023|      20|211.88|
|     14|        11|2022|     321|483.95|
|     15|        24|2022|     264|276.25|
|     16|        21|2022|     494|214.62|
|     17|         4|2022|     400|286.42|
|     18|         8|2022|     142|290.08|
|     19|        24|2021|     460|367.17|
|     20|         3|2020|     371| 68.21|
+-------+----------+----+--------+

In [113]:
product_df = spark.createDataFrame(product)

In [114]:
product_df.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|   Product_1|
|         2|   Product_2|
|         3|   Product_3|
|         4|   Product_4|
|         5|   Product_5|
|         6|   Product_6|
|         7|   Product_7|
|         8|   Product_8|
|         9|   Product_9|
|        10|  Product_10|
|        11|  Product_11|
|        12|  Product_12|
|        13|  Product_13|
|        14|  Product_14|
|        15|  Product_15|
|        16|  Product_16|
|        17|  Product_17|
|        18|  Product_18|
|        19|  Product_19|
|        20|  Product_20|
+----------+------------+
only showing top 20 rows



In [115]:
product_df.createOrReplaceTempView("products")

In [123]:
spark.sql(
"""
  with first as
  (SELECT product_id,min(year) as first_year
  from sales 
  group by product_id
  )

    SELECT  p.product_name,
        s.year,
        s.quantity,
        s.price from sales s
    join first f 
        on s.year = f.first_year 
        AND s.product_id = f.product_id
    Join products p 
        on s.product_id = p.product_id
    ORDER BY 
        p.product_name ASC , 
        s.sale_id ASC;

    
"""
).show()

+------------+----+--------+------+
|product_name|year|quantity| price|
+------------+----+--------+------+
|    Keyboard|2019|      50| 49.99|
|      Laptop|2020|     120| 19.99|
|      Laptop|2020|      80| 18.49|
|     Monitor|2021|      40| 99.99|
|       Mouse|2020|     100| 29.99|
|       Mouse|2020|      60| 31.99|
|     Printer|2023|     150|149.99|
|   Product_1|2019|     380|416.53|
|  Product_10|2019|     122|203.11|
|  Product_11|2022|     321|483.95|
|  Product_12|2019|     322|123.11|
|  Product_12|2019|      53|465.81|
|  Product_12|2019|     432|211.24|
|  Product_13|2019|      69|183.37|
|  Product_14|2019|     487| 87.28|
|  Product_14|2019|     179| 48.76|
|  Product_15|2019|     145|297.98|
|  Product_15|2019|     272|349.23|
|  Product_16|2020|     271| 50.39|
|  Product_17|2020|     397|320.14|
+------------+----+--------+------+
only showing top 20 rows



# Practice Problem: Employee First Project Assignment

In [124]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -------------------------
# Employees
# -------------------------
n_employees = 30

employees = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "employee_name": [f"Employee_{i}" for i in range(1, n_employees + 1)]
})

# -------------------------
# Assignments
# -------------------------
n_rows = 200

assignments = pd.DataFrame({
    "assignment_id": np.arange(1, n_rows + 1),
    "employee_id": np.random.randint(1, n_employees + 1, n_rows),
    "project_id": np.random.randint(100, 151, n_rows),
    "assignment_year": np.random.choice(
        [2019, 2020, 2021, 2022, 2023],
        n_rows
    ),
    "hours_allocated": np.random.randint(20, 301, n_rows)
})

# Employees with multiple assignments in first year
extra_assignments = pd.DataFrame({
    "assignment_id": np.arange(n_rows + 1, n_rows + 13),
    "employee_id": [
        101,101,101,
        102,102,
        103,103,
        104,
        105,105,
        106,106
    ],
    "project_id": [
        501,502,503,
        504,505,
        506,507,
        508,
        509,510,
        511,512
    ],
    "assignment_year": [
        2020,2020,2021,
        2019,2020,
        2021,2022,
        2023,
        2020,2020,
        2019,2021
    ],
    "hours_allocated": [
        120,80,140,
        60,100,
        90,110,
        150,
        70,95,
        50,75
    ]
})

extra_employees = pd.DataFrame({
    "employee_id": [101,102,103,104,105,106],
    "employee_name": [
        "Alice",
        "Bob",
        "Charlie",
        "David",
        "Emma",
        "Frank"
    ]
})

employees = pd.concat([employees, extra_employees], ignore_index=True)
assignments = pd.concat([assignments, extra_assignments], ignore_index=True)

print(employees.head())
print()
print(assignments.head())

   employee_id employee_name
0            1    Employee_1
1            2    Employee_2
2            3    Employee_3
3            4    Employee_4
4            5    Employee_5

   assignment_id  employee_id  project_id  assignment_year  hours_allocated
0              1            7         138             2019              287
1              2           20         144             2022              202
2              3           29         114             2022               32
3              4           15         142             2023              298
4              5           11         128             2023              236


In [125]:
emp_df = spark.createDataFrame(employees)

asgn_df = spark.createDataFrame(assignments)

In [126]:
emp_df.createOrReplaceTempView("emp")
asgn_df.createOrReplaceTempView("asgn")

In [131]:
spark.sql(
"""
    with first as (
    SELECT
        employee_id,
        min(assignment_year) as first_year
    from asgn 
        group by employee_id
    )

    SELECT e.employee_name,a.assignment_year,a.project_id,a.hours_allocated
    from 
        asgn a 
    join 
        first f
    ON 
        a.employee_id = f.employee_id 
    AND 
        a.assignment_year = f.first_year
    JOIN 
        emp e 
    ON 
        e.employee_id = a.employee_id



"""
).show()

+-------------+---------------+----------+---------------+
|employee_name|assignment_year|project_id|hours_allocated|
+-------------+---------------+----------+---------------+
|   Employee_1|           2020|       116|            269|
|   Employee_1|           2020|       140|            275|
|   Employee_1|           2020|       140|             61|
|   Employee_2|           2020|       134|            102|
|   Employee_2|           2020|       105|             99|
|   Employee_2|           2020|       110|             44|
|   Employee_3|           2019|       121|             73|
|   Employee_4|           2019|       135|            183|
|   Employee_5|           2019|       134|            152|
|   Employee_5|           2019|       134|            124|
|   Employee_5|           2019|       136|            184|
|   Employee_6|           2019|       128|            222|
|   Employee_7|           2019|       146|            185|
|   Employee_7|           2019|       101|             8

In [133]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_rows = 500

orders = pd.DataFrame({
    "order_id": np.arange(1, n_rows + 1),
    "customer_id": np.random.randint(1, 51, n_rows),
    "product_id": np.random.randint(100, 151, n_rows),
    "store_id": np.random.randint(1, 11, n_rows),
    "salesperson_id": np.random.randint(1, 16, n_rows),
    "department_id": np.random.randint(1, 6, n_rows),
    "employee_id": np.random.randint(1, 31, n_rows),
    "quantity": np.random.randint(1, 20, n_rows),
    "amount": np.round(np.random.uniform(100, 5000, n_rows), 2),
    "order_date": np.random.choice(
        pd.date_range("2024-01-01", "2024-06-30").strftime("%Y-%m-%d"),
        n_rows
    )
})

# Inject predictable records
extra = pd.DataFrame({
    "order_id": np.arange(501, 513),
    "customer_id": [101,101,101,102,102,103,103,104,104,105,105,106],
    "product_id": [201,202,201,203,204,205,206,207,208,209,210,211],
    "store_id": [1,1,2,3,3,4,5,6,6,7,8,9],
    "salesperson_id": [20]*12,
    "department_id": [1]*12,
    "employee_id": [50]*12,
    "quantity": [5,8,2,10,4,6,7,9,3,8,2,1],
    "amount": [500,800,200,900,450,600,700,900,350,850,250,100],
    "order_date": [
        "2024-01-10",
        "2024-01-10",
        "2024-02-10",
        "2024-03-12",
        "2024-03-12",
        "2024-04-05",
        "2024-04-05",
        "2024-05-08",
        "2024-05-08",
        "2024-06-15",
        "2024-06-15",
        "2024-06-20"
    ]
})

orders = pd.concat([orders, extra], ignore_index=True)

print(orders.head())
print("\nShape:", orders.shape)

   order_id  customer_id  product_id  store_id  salesperson_id  department_id  \
0         1           39         147         1              11              2   
1         2           29         120         3              10              2   
2         3           15         138        10               7              4   
3         4           43         135         9               1              1   
4         5            8         132         5              14              3   

   employee_id  quantity   amount  order_date  
0           22         6  2211.83  2024-03-08  
1            1         4  1662.45  2024-05-01  
2            4        16  2952.77  2024-01-15  
3           14        15  1918.74  2024-06-13  
4            1        15  3045.26  2024-05-08  

Shape: (512, 10)


In [134]:
df = spark.createDataFrame(orders)

In [135]:
display(df)

,order_id,customer_id,product_id,store_id,salesperson_id,department_id,employee_id,quantity,amount,order_date
0,1,39,147,1,11,2,22,6,2211.83,2024-03-08
1,2,29,120,3,10,2,1,4,1662.45,2024-05-01
2,3,15,138,10,7,4,4,16,2952.77,2024-01-15
3,4,43,135,9,1,1,14,15,1918.74,2024-06-13
4,5,8,132,5,14,3,1,15,3045.26,2024-05-08


In [136]:
df.groupBy("customer_id").agg(sum(col("amount") * col("quantity"))).show()

+-----------+------------------------+
|customer_id|sum((amount * quantity))|
+-----------+------------------------+
|         29|      220405.28999999998|
|         26|               425007.88|
|         19|      276409.33999999997|
|         22|               264885.71|
|          7|      301042.35000000003|
|         34|               160274.96|
|         50|      175803.91999999998|
|         43|      112456.70999999999|
|         31|       93127.48000000001|
|         39|       488353.6400000001|
|         25|               293959.78|
|          6|      224799.72999999998|
|          9|               278926.52|
|         27|               207869.13|
|         17|               205250.77|
|         41|      143014.47999999998|
|         33|      418574.20000000007|
|         28|               362679.97|
|          5|               137072.13|
|          1|               284897.96|
+-----------+------------------------+
only showing top 20 rows

